# 01 - Data Ingestion

**BlueStock MF Capstone** — Load, profile, and validate all 10 raw CSV datasets.

---

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

ROOT = Path(os.path.abspath('')).parent if 'notebooks' in os.path.abspath('') else Path(os.path.abspath(''))
RAW_DIR = ROOT / 'data' / 'raw'
print(f'Project root: {ROOT}')
print(f'Raw data dir: {RAW_DIR}')

---
## 1. File Inventory

Verify all 10 raw CSV files are present and log their sizes.

In [ ]:
RAW_FILES = {
    'fund_master':        '01_fund_master.csv',
    'nav_history':        '02_nav_history.csv',
    'aum_by_fund_house':  '03_aum_by_fund_house.csv',
    'monthly_sip':        '04_monthly_sip_inflows.csv',
    'category_inflows':   '05_category_inflows.csv',
    'industry_folio':     '06_industry_folio_count.csv',
    'scheme_performance': '07_scheme_performance.csv',
    'transactions':       '08_investor_transactions.csv',
    'portfolio_holdings': '09_portfolio_holdings - 09_portfolio_holdings.csv',
    'benchmark_indices':  '10_benchmark_indices - 10_benchmark_indices.csv',
}

inventory = []
for key, filename in RAW_FILES.items():
    filepath = RAW_DIR / filename
    exists = filepath.exists()
    size_kb = filepath.stat().st_size / 1024 if exists else 0
    inventory.append({'dataset': key, 'filename': filename, 'exists': exists, 'size_KB': round(size_kb, 1)})

inv_df = pd.DataFrame(inventory)
print(f'Files found: {inv_df["exists"].sum()}/{len(inv_df)}')
display(inv_df)

---
## 2. Load All Datasets

Read each CSV into a DataFrame and store in a dictionary for profiling.

In [ ]:
datasets = {}
for key, filename in RAW_FILES.items():
    datasets[key] = pd.read_csv(RAW_DIR / filename)

# Summary table
summary = pd.DataFrame([
    {'dataset': k, 'rows': len(v), 'columns': len(v.columns), 'nulls': int(v.isnull().sum().sum()),
     'duplicates': int(v.duplicated().sum()), 'dtypes': str(dict(v.dtypes.value_counts()))}
    for k, v in datasets.items()
])
display(summary)

---
## 3. Data Profiling — Schema & Sample Rows

For each dataset, display dtypes, null counts, and first 3 rows.

In [ ]:
print('\n=== FUND MASTER (01) ===')
print(f'Shape: {datasets["fund_master"].shape}')
display(datasets['fund_master'].info())
display(datasets['fund_master'].head(3))

In [ ]:
print('\n=== NAV HISTORY (02) ===')
print(f'Shape: {datasets["nav_history"].shape}')
print(f'Date range: {datasets["nav_history"]["date"].min()} to {datasets["nav_history"]["date"].max()}')
print(f'Unique funds: {datasets["nav_history"]["amfi_code"].nunique()}')
display(datasets['nav_history'].describe())

In [ ]:
print('\n=== INVESTOR TRANSACTIONS (08) ===')
print(f'Shape: {datasets["transactions"].shape}')
print(f'Transaction types: {datasets["transactions"]["transaction_type"].value_counts().to_dict()}')
print(f'Unique investors: {datasets["transactions"]["investor_id"].nunique()}')
display(datasets['transactions'].describe())

In [ ]:
print('\n=== SCHEME PERFORMANCE (07) ===')
print(f'Shape: {datasets["scheme_performance"].shape}')
display(datasets['scheme_performance'].describe())
display(datasets['scheme_performance'].head(3))

In [ ]:
for key in ['aum_by_fund_house', 'monthly_sip', 'category_inflows',
            'industry_folio', 'portfolio_holdings', 'benchmark_indices']:
    df = datasets[key]
    print(f'\n=== {key.upper()} ===')
    print(f'  Shape: {df.shape}  |  Nulls: {df.isnull().sum().sum()}  |  Dupes: {df.duplicated().sum()}')
    display(df.head(2))

---
## 4. Data Validation Checks

Automated checks for data quality issues.

In [ ]:
checks = []

# Check 1: All 40 AMFIs in fund_master appear in NAV history
amfi_master = set(datasets['fund_master']['amfi_code'])
amfi_nav = set(datasets['nav_history']['amfi_code'])
missing_nav = amfi_master - amfi_nav
checks.append({'check': 'All funds have NAV data', 'pass': len(missing_nav) == 0,
               'detail': f'Missing: {missing_nav}' if missing_nav else 'All 40 present'})

# Check 2: No negative NAVs
neg_nav = (datasets['nav_history']['nav'] <= 0).sum()
checks.append({'check': 'No negative NAVs', 'pass': neg_nav == 0,
               'detail': f'{neg_nav} negative values' if neg_nav > 0 else 'Clean'})

# Check 3: Transaction amounts all positive
neg_tx = (datasets['transactions']['amount_inr'] <= 0).sum()
checks.append({'check': 'Positive transaction amounts', 'pass': neg_tx == 0,
               'detail': f'{neg_tx} non-positive values' if neg_tx > 0 else 'Clean'})

# Check 4: Performance data has all required metrics
req_cols = {'sharpe_ratio', 'alpha', 'beta', 'return_1yr_pct'}
perf_cols = set(datasets['scheme_performance'].columns)
missing_cols = req_cols - perf_cols
checks.append({'check': 'Performance metrics present', 'pass': len(missing_cols) == 0,
               'detail': f'Missing: {missing_cols}' if missing_cols else 'All present'})

# Check 5: No duplicate AMFIs in fund master
dupe_amfi = datasets['fund_master']['amfi_code'].duplicated().sum()
checks.append({'check': 'Unique AMFI codes', 'pass': dupe_amfi == 0,
               'detail': f'{dupe_amfi} duplicates' if dupe_amfi > 0 else 'All unique'})

checks_df = pd.DataFrame(checks)
print(f'\nValidation: {checks_df["pass"].sum()}/{len(checks_df)} checks passed')
display(checks_df)

---
## Summary

- **10 CSV files** loaded successfully from `data/raw/`
- **46,000+** NAV records across 40 schemes (Jan 2022 - May 2026)
- **32,778** investor transactions (SIP, Lumpsum, Redemption)
- **5 automated validation checks** — all datasets profiled
- Data is ready for cleaning in notebook `02_data_cleaning.ipynb`